In [ ]:
# --- Config: paths & helpers (auto-inserted) ---
from pathlib import Path
import os, json
import pandas as pd

# Assume the notebook is opened from repository root (safe for most cases)
REPO = Path.cwd()
os.environ["PYTHONPATH"] = str(REPO)

# Core paths
META = REPO / "meta" / "master_metadata.csv"
LOGS = REPO / "logs"
LOGS_EF = REPO / "logs_ef"
LOGS_VOL = REPO / "logs_vol"
REPORTS = REPO / "reports"
REPORTS_SEG = REPO / "reports_seg"
REPORTS.mkdir(exist_ok=True); REPORTS_SEG.mkdir(exist_ok=True)

# --- Loaders ---
def load_seg_summaries():
    out = {}
    if LOGS.exists():
        for p in LOGS.glob("cv_seg_*_summary.json"):
            try:
                with open(p) as f:
                    out[p.stem] = json.load(f)
            except Exception:
                pass
    return out

def load_clf_summaries():
    out = {}
    p = LOGS / "cv_cls_summary.csv"
    if p.exists():
        out["all"] = pd.read_csv(p)
    p = LOGS_EF / "cv_cls_summary.csv"
    if p.exists():
        out["ef"] = pd.read_csv(p)
    p = LOGS_VOL / "cv_cls_summary.csv"
    if p.exists():
        out["vol"] = pd.read_csv(p)
    return out


## 5) Per-fold ROC & Confusion Matrices (if logged)

In [ ]:
from pathlib import Path
import IPython.display as disp

for prefix in ['cv_image_fold','cv_tabular_fold']:
    found=False
    for i in range(1, 51):
        roc = logs_dir / f"{prefix}{i}.roc.png"
        cm  = logs_dir / f"{prefix}{i}.cm.png"
        if roc.exists() or cm.exists():
            found=True
            print(f"Assets for {prefix}{i}")
            if roc.exists(): disp.display(disp.Image(filename=str(roc)))
            if cm.exists():  disp.display(disp.Image(filename=str(cm)))
    if not found:
        print(f"No PNG assets found for {prefix}*")

## 6) Precision–Recall Curves (classification, per class if available)

In [ ]:
from pathlib import Path
import IPython.display as disp
for prefix in ['cv_image_three_fold','cv_image_binary_fold']:
    found=False
    for i in range(1, 51):
        # per-class PR assets
        base = logs_dir / f"{prefix}{i}"
        # 3-class variants
        for k in range(3):
            prp = base.with_suffix(f'.class{k}.pr.png')
            if prp.exists():
                found=True
                print(f'PR: {prp.name}')
                disp.display(disp.Image(filename=str(prp)))
        # binary variant
        pr_bin = base.with_suffix('.pr.png')
        if pr_bin.exists():
            found=True
            print(f'PR: {pr_bin.name}')
            disp.display(disp.Image(filename=str(pr_bin)))
    if not found:
        print(f'No PR assets found for {prefix}*')

## 7) Ablation results

In [ ]:
abl = logs_dir / 'ablation_cls.csv'
if abl.exists():
    import pandas as pd
    df = pd.read_csv(abl)
    display(df)
else:
    print('No ablation file found. Run scripts/ablate_classification.py.')

## 8) Per-class Dice (ACDC multiclass)

In [ ]:
perclass = logs_dir / 'cv_seg_acdc_multiclass_perclass.csv'
if perclass.exists():
    import pandas as pd
    df = pd.read_csv(perclass)
    display(df)
else:
    print('No per-class Dice CSV yet. Run seg_cv.py with --acdc-multiclass.')

## 9) Macro/weighted AUC & AP (classification summary)

In [ ]:
import json
for lab in ['three','binary']:
    fp = logs_dir / f'cv_image_{lab}_summary.json'
    if fp.exists():
        s = json.loads(fp.read_text())
        keys = ['AUC_macro_mean','AUC_macro_std','AUC_weighted_mean','AUC_weighted_std','AP_macro_mean','AP_macro_std','AP_weighted_mean','AP_weighted_std']
        out = {k: s.get(k, None) for k in keys}
        print(lab, out)
    else:
        print('Summary not found for', lab)